# PINNAR: Planetary Resource Mapping on PDS Subsets

This notebook demonstrates the end-to-end inference workflow of the Physics-Informed Neural Network for Planetary Resource Mapping (PINNAR). It loads a mock (for demonstration purposes) subset of hyperspectral data, runs the pre-trained PINNAR model, and visualizes the resulting mineral abundance and uncertainty maps.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# Add src to path
sys.path.append(os.path.abspath('..'))
from src.hybrid_model import SpatialSpectralAbundanceModel
from src.postprocessing import ThresholdMRF

## 1. Data Loading (Mock Dataset)

Here we generate a mock hyperspectral image (e.g., $100 \times 100$ pixels with $50$ spectral bands) to represent a small region of interest from a PDS dataset like CRISM or M3.

In [ ]:
# Generate mock hyperspectral cube
width, height, num_bands = 100, 100, 50
mock_hyperspectral_data = torch.rand((1, num_bands, height, width))
print(f"Mock data shape: {mock_hyperspectral_data.shape}")

## 2. Model Initialization and Inference

We instantiate the spatial-spectral hybrid model. In a real scenario, we would load weights via `model.load_state_dict(...)`.

In [ ]:
num_endmembers = 5
model = SpatialSpectralAbundanceModel(num_bands=num_bands, num_endmembers=num_endmembers)

# Set to evaluation mode
model.eval()

# Perform inference
with torch.no_grad():
    # Output shape: (Batch, num_endmembers, height, width)
    predicted_abundances = model(mock_hyperspectral_data)
    
print(f"Predicted abundances shape: {predicted_abundances.shape}")

## 3. Visualization

We extract the abundance map for the first mineral endmember and visualize it.

In [ ]:
# Extract first endmember (e.g., Olivine)
endmember_idx = 0
abundance_map = predicted_abundances[0, endmember_idx, :, :].numpy()

plt.figure(figsize=(8, 6))
plt.imshow(abundance_map, cmap='viridis')
plt.colorbar(label='Predicted Abundance')
plt.title('Mineral Abundance Map (Mock)')
plt.axis('off')
plt.show()